In [1]:
import pandas
import torch
import torchvision
import matplotlib.pyplot as plt
import torchvision.transforms.v2

import zigzag.utils
from zigzag.pipelines.validate import train_validate, validate_pretrained

In [2]:
def hidden_state_layers(model: torch.nn.Module) -> list[tuple[str, tuple[int, ...]]]:
    if isinstance(model, torchvision.models.VisionTransformer):
        return [(f'encoder.layers.{i}', (i,)) for i, _ in enumerate(model.encoder.layers)]
    if isinstance(model, torchvision.models.ResNet):
        return [
            (f'layer{stage}.{block}', (stage, block))
            for stage in range(1, 5)
            for block, _ in enumerate(getattr(model, f'layer{stage}'))
        ]
    raise TypeError(f'Unsupported architecture: {type(model)}')

class ResidualBlockBypass(torch.nn.Module):
    def __init__(self, block: torch.nn.Module):
        super().__init__()
        self.downsample = block.downsample

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x if self.downsample is None else self.downsample(x)

def remove_hidden_state_layer(model: torch.nn.Module, location: tuple[int, ...]) -> None:
    if isinstance(model, torchvision.models.VisionTransformer):
        model.encoder.layers[location[0]] = torch.nn.Identity()
    elif isinstance(model, torchvision.models.ResNet):
        blocks = getattr(model, f'layer{location[0]}')
        blocks[location[1]] = ResidualBlockBypass(blocks[location[1]])
    else:
        raise TypeError(f'Unsupported architecture: {type(model)}')

In [5]:
def final_accuracy(history: pandas.DataFrame) -> float:
    for column in ('Accuracy', 'accuracy'):
        if column in history:
            return float(history[column].iloc[-1])
    raise KeyError(f'No accuracy column in training history: {list(history.columns)}')

def run_ablations(model_name, make_model) -> pandas.DataFrame:
    _, transforms = make_model(torch.nn.Identity())
    transforms = torchvision.transforms.v2.Compose([torchvision.transforms.v2.Grayscale(num_output_channels=3), transforms()])
    train_ds = torchvision.datasets.MNIST("mnist", train=True, download=True, transform=transforms)
    test_ds = torchvision.datasets.MNIST("mnist", train=False, download=True, transform=transforms)
    train_y = train_ds.targets
    test_y = test_ds.targets

    dumper = zigzag.utils.UniversalDumper(f'ablation_results/mnist/{model_name}')

    model, _ = make_model(torch.nn.Identity())
    pretrained_dumper = dumper.make_subdumper('pretrained')
    validate_pretrained(model, train_ds, train_y, test_ds, test_y, pretrained_dumper)

    finetuned_dumper = dumper.make_subdumper('finetuned')
    model, _ = make_model(pretrained_dumper.get_dump('trained_head'))
    train_validate(model, train_ds, test_ds, finetuned_dumper)
    baseline_accuracy = final_accuracy(finetuned_dumper.get_dump('train_model_history'))

    results = []
    for index, (layer_name, location) in enumerate(hidden_state_layers(make_model(torch.nn.Identity())[0]), start=1):
        print(f"Trying layer {layer_name}, {location}")
        layer_dumper = dumper.make_subdumper(f'{index:02d}_{layer_name}')
        ablated_model = finetuned_dumper.get_dump("trained_model")
        remove_hidden_state_layer(ablated_model, location)
        train_validate(ablated_model, train_ds, test_ds, layer_dumper)
        accuracy = final_accuracy(layer_dumper.get_dump('train_model_history'))
        results.append({
            'model': model_name,
            'layer_index': index,
            'layer': layer_name,
            'baseline_accuracy': baseline_accuracy,
            'ablated_accuracy': accuracy,
            'accuracy_degradation': baseline_accuracy - accuracy
        })
    results = pandas.DataFrame(results)
    dumper.save_dump(results, 'summary')
    return results

In [6]:
def make_vit_b_32(head: torch.nn.Module):
    model = torchvision.models.vit_b_32(num_classes=1000, weights=torchvision.models.ViT_B_32_Weights.DEFAULT)
    model.heads = head
    return model, torchvision.models.ViT_B_32_Weights.DEFAULT.transforms

run_ablations("vit_b_32", make_vit_b_32)

Got the result from ablation_results/mnist/vit_b_32/pretrained\train_embeddings.pt
Got the result from ablation_results/mnist/vit_b_32/pretrained\test_embeddings.pt
Got the result from ablation_results/mnist/vit_b_32/pretrained\train_head_history.csv
Got the result from ablation_results/mnist/vit_b_32/pretrained\trained_head.pth


d:\HSE\CourseProject-Mag-1\zigzag\utils\dumper.py:116: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(file)


Got the result from ablation_results/mnist/vit_b_32/finetuned\train_model_history.csv
Got the result from ablation_results/mnist/vit_b_32/finetuned\train_model_history.csv
Trying layer encoder.layers.0, (0,)
Got the result from ablation_results/mnist/vit_b_32/finetuned\trained_model.pth


Training:  40%|████      | 4/10 [48:17<1:12:26, 724.42s/it, Accuracy=0.0892, AUC-ROC=0.496, Precision=0.00892, Recall=0.1, F1-score=0.0164, TOP-2 Accuracy=0.203, TOP-3 Accuracy=0.298, TOP-5 Accuracy=0.495, TOP-7 Accuracy=0.701, TOP-9 Accuracy=0.899]


KeyboardInterrupt: 

In [ ]:
def make_vit_l_16(head: torch.nn.Module):
    model = torchvision.models.vit_l_16(num_classes=1000, weights=torchvision.models.ViT_L_16_Weights.DEFAULT)
    model.heads = head
    return model, torchvision.models.ViT_L_16_Weights.DEFAULT.transforms

run_ablations("vit_l_16", make_vit_l_16)

In [ ]:
def make_resnet34(head: torch.nn.Module):
    model = torchvision.models.resnet34(num_classes=1000, weights=torchvision.models.ResNet34_Weights.DEFAULT)
    model.fc = head
    return model, torchvision.models.ResNet34_Weights.DEFAULT.transforms

run_ablations("resnet34", make_resnet34)

In [ ]:
fig, axes = plt.subplots(len(MODEL_CONFIGS), 1, figsize=(15, 11), constrained_layout=True)
for axis, (model_name, model_results) in zip(axes, results.groupby('model', sort=False)):
    axis.bar(model_results['layer'], model_results['accuracy_degradation'], color='tab:red')
    axis.axhline(0, color='black', linewidth=0.8)
    axis.set_title(f'{model_name}: final MNIST accuracy degradation after one-layer ablation')
    axis.set_ylabel('baseline accuracy − ablated accuracy')
    axis.tick_params(axis='x', rotation=45)
axes[-1].set_xlabel('removed block (same order as yield_hidden_states)')
plt.show()
